# Module 3: Estimation and Hypothesis Testing

In this module, we move beyond point estimates and start exploring interval estimation (Confidence Intervals) and the foundational framework of hypothesis testing.

### Learning Objectives:
- Construct and interpret Confidence Intervals for a population mean (both known and unknown variance).
- Calculate the required sample size for a desired margin of error.
- Understand the logic of hypothesis testing (H₀, H₁, p-values, errors).
- Conduct one-sample hypothesis tests (Z-test and t-test).

In [1]:
# Setup: Importing necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import math

sns.set_style("whitegrid")
np.random.seed(42)
print("Setup Complete")

Setup Complete


## Section 1: Interval Estimation (Confidence Intervals)

A point estimate (like the sample mean $\bar{x}$) gives a single value, but it's almost certainly not exactly equal to the population parameter ($\mu$). A **Confidence Interval (CI)** provides a range of plausible values for the parameter.

A CI is constructed as: Point Estimate ± Margin of Error.

### 1.1 Confidence Interval for Mean (σ known - Z-interval)

When the population standard deviation (σ) is known (which is rare in practice), we can use the Z-distribution (Standard Normal distribution).

Formula: $ \bar{x} \pm Z_{\alpha/2} \cdot SE $, where $SE = \frac{\sigma}{\sqrt{n}}$

In [2]:
# Scenario: We are measuring the lifespan of a specific brand of light bulb.
# We know from the manufacturer that the population std dev is σ=100 hours.
# We take a sample of 40 bulbs (n=40) and find the sample mean lifespan is x̄=780 hours.
sigma = 100 # Population std dev (known)
n = 40      # Sample size
x_bar = 780  # Sample mean

# We want a 95% Confidence Interval.
confidence_level = 0.95
alpha = 1 - confidence_level # Significance level (0.05)

# 1. Find the critical Z-value (Z_α/2)
# stats.norm.ppf (Percent Point Function) gives the Z-score for a given cumulative probability.
# For 95% confidence, we look up the value corresponding to 1 - α/2 = 1 - 0.025 = 0.975
z_critical = stats.norm.ppf(1 - alpha/2)
print(f"Critical Z-value for 95% CI: {z_critical:.3f} (Commonly known as 1.96)")

# 2. Calculate the Standard Error (SE = σ/√n)
standard_error = sigma / math.sqrt(n)
print(f"Standard Error: {standard_error:.3f}")

# 3. Calculate the Margin of Error (ME = Z_critical * SE)
margin_of_error = z_critical * standard_error
print(f"Margin of Error: {margin_of_error:.3f}")

# 4. Calculate the Confidence Interval
ci_lower = x_bar - margin_of_error
ci_upper = x_bar + margin_of_error

print(f"\n95% Confidence Interval (Z): ({ci_lower:.2f}, {ci_upper:.2f})")

# Interpretation: We are 95% confident that the true population mean lifespan of the bulbs lies between 748.99 and 811.01 hours.

Critical Z-value for 95% CI: 1.960 (Commonly known as 1.96)
Standard Error: 15.811
Margin of Error: 30.990

95% Confidence Interval (Z): (749.01, 810.99)


### 1.2 Confidence Interval for Mean (σ unknown - t-interval)

In most real-world scenarios, the population standard deviation (σ) is unknown. We estimate it using the sample standard deviation (s).

When using 's' instead of 'σ', we must use the **Student's t-distribution**. The t-distribution has heavier tails than the Z-distribution, accounting for the extra uncertainty from estimating σ.

Formula: $ \bar{x} \pm t_{\alpha/2, df} \cdot SE $, where $SE = \frac{s}{\sqrt{n}}$ and $df = n-1$ (degrees of freedom).

In [3]:
# Scenario: We measure the heights of a sample of 20 students (n=20).
# Sample mean x̄=170cm, Sample std dev s=8cm. σ is unknown.

n = 20
x_bar = 170
s = 8 # Sample std dev
confidence_level = 0.95
alpha = 1 - confidence_level
df = n - 1 # Degrees of freedom

# 1. Find the critical t-value (t_α/2, df)
# We use stats.t.ppf instead of stats.norm.ppf
t_critical = stats.t.ppf(1 - alpha/2, df=df)
print(f"Critical t-value (df={df}): {t_critical:.3f}")
# Note: The t-value (2.093) is larger than the Z-value (1.96), leading to a wider interval because we have less certainty.

# 2. Calculate the Standard Error (SE = s/√n)
standard_error = s / math.sqrt(n)

# 3. Calculate the Margin of Error (ME = t_critical * SE)
margin_of_error = t_critical * standard_error
print(f"Margin of Error: {margin_of_error:.3f}")

# 4. Calculate the Confidence Interval
ci_lower = x_bar - margin_of_error
ci_upper = x_bar + margin_of_error

print(f"\n95% Confidence Interval (t): ({ci_lower:.2f}, {ci_upper:.2f})")

Critical t-value (df=19): 2.093
Margin of Error: 3.744

95% Confidence Interval (t): (166.26, 173.74)


### 1.3 Calculating CI using `scipy.stats` directly

Python libraries can calculate these intervals automatically from raw data.

In [4]:
# Generate some sample data
data = np.random.normal(50, 5, 30)

# Calculate 95% CI using t-distribution (the standard approach as σ is typically unknown)
confidence_level = 0.95
n = len(data)
mean = np.mean(data)
# stats.sem calculates the standard error (s/√n) automatically from the data
std_error = stats.sem(data)

# stats.t.interval calculates the interval based on confidence level, df, location (mean), and scale (SE)
ci = stats.t.interval(confidence_level, df=n-1, loc=mean, scale=std_error)

print(f"Sample Mean: {mean:.2f}")
print(f"95% Confidence Interval (Automatic): ({ci[0]:.2f}, {ci[1]:.2f})")

Sample Mean: 49.06
95% Confidence Interval (Automatic): (47.38, 50.74)


## Section 2: Determining Sample Size

Often, we want to determine the necessary sample size (n) required to achieve a specific Margin of Error (ME) at a given confidence level.

We rearrange the Margin of Error formula (using Z, assuming we have an estimate for σ):
$ ME = Z_{\alpha/2} \frac{\sigma}{\sqrt{n}} $

Solving for n: $ n = (\frac{Z_{\alpha/2} \cdot \sigma}{ME})^2 $

In [ ]:
# Scenario: We want to estimate the average battery life of a new smartphone.
# We estimate the population std dev σ=2 hours (based on similar models).
# We want the Margin of Error (ME) to be 0.5 hours, with 99% confidence.

ME_desired = 0.5
sigma_est = 2
confidence_level = 0.99
alpha = 1 - confidence_level

# Find Z critical value for 99% confidence
z_critical = stats.norm.ppf(1 - alpha/2)
print(f"Critical Z-value for 99% CI: {z_critical:.3f}")

# Calculate required sample size
n_required = ((z_critical * sigma_est) / ME_desired)**2

print(f"Required sample size: {n_required:.2f}")
# We ALWAYS round UP the sample size to ensure the desired ME is achieved.
print(f"Required sample size (rounded up): {math.ceil(n_required)}")

## Section 3: The Logic of Hypothesis Testing

Hypothesis testing is a formal procedure for investigating our ideas about the world using statistics. It is essentially a proof by contradiction.

### 3.1 Key Concepts

1.  **Null Hypothesis (H₀):** The default assumption; a statement of no effect or no difference. We seek evidence against this.
2.  **Alternative Hypothesis (H₁ or Ha):** What we want to prove; a statement that contradicts H₀.
3.  **Significance Level (α):** The threshold for deciding if evidence is strong enough to reject H₀. Commonly 0.05.
4.  **Test Statistic:** A value calculated from the sample data (e.g., t-statistic, Z-statistic).
5.  **P-value:** The probability of observing a test statistic as extreme as (or more extreme than) the one calculated, *assuming H₀ is true*.

### 3.2 Decision Rule

We compare the p-value to α:

- If **p-value < α**: Reject H₀. The result is statistically significant.
- If **p-value ≥ α**: Fail to reject H₀. The result is not statistically significant.

### 3.3 Type I and Type II Errors

| Decision | H₀ is True | H₀ is False |
| :--- | :---: | :--- |
| **Reject H₀** | Type I Error (α) (False Positive) | Correct Decision (Power) |
| **Fail to Reject H₀** | Correct Decision | Type II Error (β) (False Negative) |

- **Type I Error (α):** Rejecting a true H₀. (e.g., Concluding a drug works when it doesn't).
- **Type II Error (β):** Failing to reject a false H₀. (e.g., Concluding a drug doesn't work when it actually does).

## Section 4: One-Sample Hypothesis Tests

We use these tests to compare the mean of a single sample to a known or hypothesized population mean.

### 4.1 One-Sample t-test

Used when the population variance (σ²) is unknown (the most common scenario). (The Z-test is used when σ² is known, following the same logic but using the Z-distribution).

In [ ]:
# Scenario: A university claims the average GPA of their graduates is 3.5 (μ₀=3.5).
# We take a random sample of 50 graduates to test this claim.

# Step 1: State the hypotheses
# H₀: μ = 3.5 (The mean GPA is 3.5)
# H₁: μ ≠ 3.5 (The mean GPA is different from 3.5) - Two-tailed test

# Step 2: Set the significance level
alpha = 0.05
mu_hypothesized = 3.5

# Generate sample data (let's assume the true mean is slightly lower)
sample_data = np.random.normal(loc=3.4, scale=0.4, size=50)

# Step 3: Calculate the test statistic and p-value
# We use scipy.stats.ttest_1samp
t_statistic, p_value = stats.ttest_1samp(a=sample_data, popmean=mu_hypothesized)

print(f"Sample Mean GPA: {np.mean(sample_data):.4f}")
print(f"T-statistic: {t_statistic:.4f}")
print(f"P-value: {p_value:.4f}")

# Step 4: Make a decision
if p_value < alpha:
    print("\nDecision: Reject H₀.")
    print("Conclusion: There is statistically significant evidence that the average GPA is different from 3.5.")
else:
    print("\nDecision: Fail to reject H₀.")
    print("Conclusion: There is insufficient evidence to conclude that the average GPA is different from 3.5.")

### 4.2 One-Tailed Tests

Sometimes we are only interested in a difference in one direction.

Example: We want to test if the average GPA is *less than* 3.5 (H₁: μ < 3.5).

In [ ]:
# H₀: μ >= 3.5
# H₁: μ < 3.5 (One-tailed, left-tailed test)

# We use the 'alternative' parameter in scipy.stats.ttest_1samp (available in recent scipy versions)
try:
    t_statistic_one_tail, p_value_one_tail = stats.ttest_1samp(
        a=sample_data,
        popmean=mu_hypothesized,
        alternative='less' # Testing if sample mean is less than popmean
    )

    print(f"P-value (One-tailed, less): {p_value_one_tail:.4f}")

    # Decision making
    if p_value_one_tail < alpha:
        print("\nDecision: Reject H₀. Evidence suggests the average GPA is significantly less than 3.5.")
    else:
        print("\nDecision: Fail to reject H₀. Insufficient evidence that the average GPA is less than 3.5.")

except TypeError:
    print("Your scipy version might not support the 'alternative' parameter. Update scipy or calculate manually.")
    # Manual calculation for one-tailed test (if H1 is < and t-stat is negative)
    if t_statistic < 0:
        p_value_one_tail = p_value / 2
        print(f"P-value (One-tailed, calculated manually): {p_value_one_tail:.4f}")
    else:
        p_value_one_tail = 1 - (p_value / 2)
        print(f"P-value (One-tailed, calculated manually): {p_value_one_tail:.4f}")
